In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.utils import resample

from sklearn.utils.class_weight import compute_class_weight

import re
import nltk
from nltk import pos_tag
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction import DictVectorizer

from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB  
from hmmlearn.hmm import GaussianHMM


from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense , Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import Model

import joblib
from scipy.sparse import hstack

import spacy
import pickle


In [62]:
!pip install hmmlearn

In [4]:
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')
nlp = spacy.load("en_core_web_sm")


[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\20100\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\20100\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


## Reading Data

In [5]:
data = pd.read_json("News_Category_Dataset_v3.json", lines=True)

In [ ]:
data.head()	

,link,headline,category,short_description,authors,date
0,https://www.huffpost.com/entry/covid-boosters-...,Over 4 Million Americans Roll Up Sleeves For O...,U.S. NEWS,Health experts said it is too early to predict...,"Carla K. Johnson, AP",2022-09-23
1,https://www.huffpost.com/entry/american-airlin...,"American Airlines Flyer Charged, Banned For Li...",U.S. NEWS,He was subdued by passengers and crew when he ...,Mary Papenfuss,2022-09-23
2,https://www.huffpost.com/entry/funniest-tweets...,23 Of The Funniest Tweets About Cats And Dogs ...,COMEDY,"""Until you have a dog you don't understand wha...",Elyse Wanshel,2022-09-23
3,https://www.huffpost.com/entry/funniest-parent...,The Funniest Tweets From Parents This Week (Se...,PARENTING,"""Accidentally put grown-up toothpaste on my to...",Caroline Bologna,2022-09-23
4,https://www.huffpost.com/entry/amy-cooper-lose...,Woman Who Called Cops On Black Bird-Watcher Lo...,U.S. NEWS,Amy Cooper accused investment firm Franklin Te...,Nina Golgowski,2022-09-22


## EDA


In [7]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 209527 entries, 0 to 209526
Data columns (total 6 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   link               209527 non-null  object        
 1   headline           209527 non-null  object        
 2   category           209527 non-null  object        
 3   short_description  209527 non-null  object        
 4   authors            209527 non-null  object        
 5   date               209527 non-null  datetime64[ns]
dtypes: datetime64[ns](1), object(5)
memory usage: 9.6+ MB


In [8]:
data.duplicated().sum()

13

In [9]:
data.drop_duplicates(inplace=True)

In [10]:
data['category'].value_counts()

category
POLITICS          35601
WELLNESS          17942
ENTERTAINMENT     17362
TRAVEL             9900
STYLE & BEAUTY     9811
PARENTING          8791
HEALTHY LIVING     6694
QUEER VOICES       6347
FOOD & DRINK       6340
BUSINESS           5992
COMEDY             5400
SPORTS             5077
BLACK VOICES       4583
HOME & LIVING      4320
PARENTS            3955
THE WORLDPOST      3664
WEDDINGS           3653
WOMEN              3571
CRIME              3562
IMPACT             3484
DIVORCE            3426
WORLD NEWS         3299
MEDIA              2944
WEIRD NEWS         2777
GREEN              2622
WORLDPOST          2579
RELIGION           2577
STYLE              2254
SCIENCE            2206
TECH               2100
TASTE              2096
MONEY              1756
ARTS               1509
ENVIRONMENT        1443
FIFTY              1401
GOOD NEWS          1398
U.S. NEWS          1377
ARTS & CULTURE     1339
COLLEGE            1144
LATINO VOICES      1130
CULTURE & ARTS     1074
EDUCATI

In [11]:
data['category'].value_counts()[:9]

category
POLITICS          35601
WELLNESS          17942
ENTERTAINMENT     17362
TRAVEL             9900
STYLE & BEAUTY     9811
PARENTING          8791
HEALTHY LIVING     6694
QUEER VOICES       6347
FOOD & DRINK       6340
Name: count, dtype: int64

In [12]:
type(data['category'].value_counts())

pandas.core.series.Series

In [13]:
(data['category'].value_counts()[10:]).sum()

84734

In [14]:
data['category'].value_counts()[10:].max()

5400

In [15]:
data['category'].value_counts()[10:].min()

1014

In [16]:
len(data['category'].value_counts())

42

In [17]:
33*((5400+1014)/2)  # size of others if we made over-sampling and under-sampling to the average

105831.0

In [18]:
33*(1014)  # size of others if we made under-sampling to the minimum (almost as the max category)

33462

## Split train test

In [19]:
train_df, test_df = train_test_split(data,test_size=0.2,random_state=42,stratify=data['category'])

In [20]:
top9_categories = list(data['category'].value_counts()[:9].index)
top9_categories

['POLITICS',
 'WELLNESS',
 'ENTERTAINMENT',
 'TRAVEL',
 'STYLE & BEAUTY',
 'PARENTING',
 'HEALTHY LIVING',
 'QUEER VOICES',
 'FOOD & DRINK']

In [21]:
def mapping_to_10(category):
    if category in top9_categories:
        return category
    else:
        return 'Other'

In [22]:
train_df['category'].value_counts()

category
POLITICS          28481
WELLNESS          14353
ENTERTAINMENT     13889
TRAVEL             7920
STYLE & BEAUTY     7849
PARENTING          7033
HEALTHY LIVING     5355
QUEER VOICES       5078
FOOD & DRINK       5072
BUSINESS           4794
COMEDY             4320
SPORTS             4062
BLACK VOICES       3666
HOME & LIVING      3456
PARENTS            3164
THE WORLDPOST      2931
WEDDINGS           2922
WOMEN              2857
CRIME              2850
IMPACT             2787
DIVORCE            2741
WORLD NEWS         2639
MEDIA              2355
WEIRD NEWS         2222
GREEN              2098
WORLDPOST          2063
RELIGION           2062
STYLE              1803
SCIENCE            1765
TECH               1680
TASTE              1677
MONEY              1405
ARTS               1207
ENVIRONMENT        1154
FIFTY              1121
GOOD NEWS          1118
U.S. NEWS          1102
ARTS & CULTURE     1071
COLLEGE             915
LATINO VOICES       904
CULTURE & ARTS      859
EDUCATI

In [23]:
train_df['category-10'] = train_df['category'].apply(mapping_to_10)
test_df['category-10'] = test_df['category'].apply(mapping_to_10)

### Resampling the "Other" category to balance the dataset

In [24]:
train_df['category'].value_counts()[9:]

category
BUSINESS          4794
COMEDY            4320
SPORTS            4062
BLACK VOICES      3666
HOME & LIVING     3456
PARENTS           3164
THE WORLDPOST     2931
WEDDINGS          2922
WOMEN             2857
CRIME             2850
IMPACT            2787
DIVORCE           2741
WORLD NEWS        2639
MEDIA             2355
WEIRD NEWS        2222
GREEN             2098
WORLDPOST         2063
RELIGION          2062
STYLE             1803
SCIENCE           1765
TECH              1680
TASTE             1677
MONEY             1405
ARTS              1207
ENVIRONMENT       1154
FIFTY             1121
GOOD NEWS         1118
U.S. NEWS         1102
ARTS & CULTURE    1071
COLLEGE            915
LATINO VOICES      904
CULTURE & ARTS     859
EDUCATION          811
Name: count, dtype: int64

In [25]:
train_df['category-10'].value_counts()

category-10
Other             72581
POLITICS          28481
WELLNESS          14353
ENTERTAINMENT     13889
TRAVEL             7920
STYLE & BEAUTY     7849
PARENTING          7033
HEALTHY LIVING     5355
QUEER VOICES       5078
FOOD & DRINK       5072
Name: count, dtype: int64

In [26]:
other =train_df[train_df['category-10']=="Other"]['category'].unique()
print(other)

['SPORTS' 'ARTS' 'ENVIRONMENT' 'FIFTY' 'WORLD NEWS' 'GREEN' 'WEDDINGS'
 'COMEDY' 'SCIENCE' 'BLACK VOICES' 'STYLE' 'PARENTS' 'DIVORCE'
 'HOME & LIVING' 'COLLEGE' 'TECH' 'WORLDPOST' 'WEIRD NEWS' 'BUSINESS'
 'RELIGION' 'IMPACT' 'WOMEN' 'GOOD NEWS' 'MEDIA' 'THE WORLDPOST' 'MONEY'
 'CULTURE & ARTS' 'EDUCATION' 'TASTE' 'CRIME' 'U.S. NEWS' 'LATINO VOICES'
 'ARTS & CULTURE']


In [27]:
non_other =train_df[train_df['category-10']!="Other"]
non_other['category-10'].value_counts() 

category-10
POLITICS          28481
WELLNESS          14353
ENTERTAINMENT     13889
TRAVEL             7920
STYLE & BEAUTY     7849
PARENTING          7033
HEALTHY LIVING     5355
QUEER VOICES       5078
FOOD & DRINK       5072
Name: count, dtype: int64

In [28]:
samples_number=train_df['category'].value_counts().min()
samples_number

811

In [29]:
balanced_data=[]
for sub in other:
    dfsub= train_df[train_df['category']==sub]
    if len(dfsub)>samples_number:
        dfsub_downsample=resample(dfsub,n_samples=samples_number,random_state=42)
    else:
        dfsun_downsample= dfsub
    balanced_data.append(dfsub_downsample)

In [30]:
train_df_balanced=pd.concat(balanced_data+[non_other])
train_df_balanced['category-10'].value_counts()

category-10
POLITICS          28481
Other             26763
WELLNESS          14353
ENTERTAINMENT     13889
TRAVEL             7920
STYLE & BEAUTY     7849
PARENTING          7033
HEALTHY LIVING     5355
QUEER VOICES       5078
FOOD & DRINK       5072
Name: count, dtype: int64

In [31]:
print("Train distribution (10 classes):")
print(train_df_balanced['category-10'].value_counts())

print("\nTest distribution (10 classes):")
print(test_df['category-10'].value_counts())

Train distribution (10 classes):
category-10
POLITICS          28481
Other             26763
WELLNESS          14353
ENTERTAINMENT     13889
TRAVEL             7920
STYLE & BEAUTY     7849
PARENTING          7033
HEALTHY LIVING     5355
QUEER VOICES       5078
FOOD & DRINK       5072
Name: count, dtype: int64

Test distribution (10 classes):
category-10
Other             18145
POLITICS           7120
WELLNESS           3589
ENTERTAINMENT      3473
TRAVEL             1980
STYLE & BEAUTY     1962
PARENTING          1758
HEALTHY LIVING     1339
QUEER VOICES       1269
FOOD & DRINK       1268
Name: count, dtype: int64


### Resample train_df_balanced

## Use class wieghts to balance "train_df_balanced"

In [32]:
y_train = train_df_balanced['category-10']
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weights_dict = dict(zip(np.unique(y_train), class_weights))
class_weights_dict

{'ENTERTAINMENT': 0.8769025847793218,
 'FOOD & DRINK': 2.401281545741325,
 'HEALTHY LIVING': 2.2743790849673204,
 'Other': 0.455079774315286,
 'PARENTING': 1.7317361012370254,
 'POLITICS': 0.42762894561286474,
 'QUEER VOICES': 2.3984442693974004,
 'STYLE & BEAUTY': 1.551700853611925,
 'TRAVEL': 1.537790404040404,
 'WELLNESS': 0.8485543092036508}

## Preprocessing Text

In [33]:
lemmatizer=WordNetLemmatizer()
stop_words=set(stopwords.words('english'))

In [34]:
train_df_balanced.columns

Index(['link', 'headline', 'category', 'short_description', 'authors', 'date',
       'category-10'],
      dtype='object')

In [35]:
train_df_balanced["text"]=  train_df_balanced["headline"] + " " + train_df_balanced["short_description"]
train_df_balanced.drop(columns=['headline', 'short_description','category-10','category','date','authors'], inplace=True)

test_df["text"]=  test_df["headline"] + " " + test_df["short_description"]
y_test = test_df['category-10']
test_df.drop(columns=['headline', 'short_description','category-10','category','date','authors'], inplace=True)

In [36]:
train_df_balanced.columns

Index(['link', 'text'], dtype='object')

In [37]:
test_df.columns

Index(['link', 'text'], dtype='object')

In [38]:
x_train = pd.DataFrame(train_df_balanced['text'])     # keeping link for scraping later
x_test = pd.DataFrame(test_df['text'])

In [39]:
def preprocess(text):
    '''function to preprocess the text data
    1)    Converting the text to lowercase
    2)    Removing non-word characters
    3)   Removing extra spaces
    4)   Tokenization
    5)   Removing stop words
    6)   Lemmatization
    7)   Joining the tokens back to form the string    
    '''
    text = text.lower()

    text=re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE) #remove URls
    text=re.sub(r'\@\w+|\#', '', text)                                    #remove special chars
    text=re.sub(r'[^a-zA-z\s]','',text)                                   #remove numbers

    tokens = word_tokenize(text)
    clean_tokens=[lemmatizer.lemmatize(word) for word in tokens if word not in stop_words and len(word)>2]
    
    return ' '.join(clean_tokens)

In [40]:
x_train["clean_text"] = x_train["text"].apply(preprocess)
x_test["clean_text"] = x_test["text"].apply(preprocess)

In [41]:
x_train

,text,clean_text
74354,Peyton Barber Declares For NFL Draft To Help H...,peyton barber declares nfl draft help homeless...
120873,Giancarlo Stanton Hit In The Face By Fastball,giancarlo stanton hit face fastball
71399,Stephen Curry Knew Exactly What To Say To Crai...,stephen curry knew exactly say craig sager las...
126337,WATCH: It's Another Nightmare Start For Brazil,watch another nightmare start brazil
82517,Carmelo Anthony Randomly Ran A Mini-Marathon M...,carmelo anthony randomly ran minimarathon midg...
...,...,...
176420,My Family Broke Up With Me Over the holidays I...,family broke holiday gifted muslim relative tr...
167182,Gwyneth Paltrow Dress Explained On Ellen: 'I K...,gwyneth paltrow dress explained ellen kinda di...
2281,White House Says Travel Bans To Remain For Now...,white house say travel ban remain delta varian...
135236,The 5 Best Bra Fit Tips -- From Men Do you hat...,best bra fit tip men hate wearing bra struggle...


## Feature Extaction

In [42]:
results = {}


### BoW

In [40]:
models={
    "SVC_BoW": SVC(class_weight=class_weights_dict),
    "MultinomialNB_BoW": MultinomialNB()
    }

In [67]:
BoW =CountVectorizer(max_features=20000) # without limiting the vocabulary size it cause crash with NN model

In [68]:
x_train_bow =BoW.fit_transform(x_train["clean_text"])
x_test_bow= BoW.transform(x_test["clean_text"])

In [44]:
print("Sparse matrix shape:", x_train_bow.shape)

Sparse matrix shape: (121793, 20000)


In [45]:
for model_name, model in models.items():    
    model.fit(x_train_bow, y_train)
    y_pred = model.predict(x_test_bow)
    accuracy=accuracy_score(y_test, y_pred)
    print(f"Results for {model_name}: accuracy={accuracy}")
    print(classification_report(y_test, y_pred))
    print('-'*50)   
    results[model_name + "BoW"] = accuracy

Results for SVC_BoW: accuracy=0.6485215855666658
                precision    recall  f1-score   support

 ENTERTAINMENT       0.54      0.75      0.63      3473
  FOOD & DRINK       0.55      0.78      0.64      1268
HEALTHY LIVING       0.30      0.44      0.36      1339
         Other       0.81      0.51      0.63     18145
     PARENTING       0.44      0.73      0.55      1758
      POLITICS       0.72      0.79      0.75      7120
  QUEER VOICES       0.75      0.67      0.70      1269
STYLE & BEAUTY       0.69      0.82      0.75      1962
        TRAVEL       0.67      0.76      0.71      1980
      WELLNESS       0.53      0.78      0.63      3589

      accuracy                           0.65     41903
     macro avg       0.60      0.70      0.64     41903
  weighted avg       0.69      0.65      0.65     41903

--------------------------------------------------
Results for MultinomialNB_BoW: accuracy=0.6145622031835429
                precision    recall  f1-score   suppor

In [46]:
results

{'SVC_BoWBoW': 0.6485215855666658, 'MultinomialNB_BoWBoW': 0.6145622031835429}

In [57]:
results['SVC_BoWBoW']=0.6485215855666658
results['MultinomialNB_BoWBoW']=0.6145622031835429

In [56]:
num_classes = len(set(y_train))  

In [ ]:
inputs = Input(shape=(x_train_bow.shape[1],), sparse=True)
x = Dense(128, activation='relu')(inputs)
x = Dense(64, activation='relu')(x)
outputs = Dense(num_classes, activation='softmax')(x)

model_NN_BoW = Model(inputs, outputs)
model_NN_BoW.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])

In [48]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

In [49]:
x_train_bow = x_train_bow.astype(np.float32)
x_test_bow  = x_test_bow.astype(np.float32)

In [50]:
y_train_int = y_train.astype('category').cat.codes  #y_train is a pandas Series convert to integer labels

model_NN_BoW.fit(x_train_bow, y_train_int, epochs=10, batch_size=32, validation_split=0.2, callbacks=[EarlyStopping(monitor='val_loss',patience=3)])

Epoch 1/10
3045/3045 ━━━━━━━━━━━━━━━━━━━━ 52s 15ms/step - accuracy: 0.6250 - loss: 1.1171 - val_accuracy: 0.7411 - val_loss: 0.8004
Epoch 2/10
3045/3045 ━━━━━━━━━━━━━━━━━━━━ 45s 15ms/step - accuracy: 0.8417 - loss: 0.4539 - val_accuracy: 0.7573 - val_loss: 0.8482
Epoch 3/10
3045/3045 ━━━━━━━━━━━━━━━━━━━━ 47s 15ms/step - accuracy: 0.9338 - loss: 0.2014 - val_accuracy: 0.7356 - val_loss: 1.2027
Epoch 4/10
3045/3045 ━━━━━━━━━━━━━━━━━━━━ 47s 15ms/step - accuracy: 0.9788 - loss: 0.0718 - val_accuracy: 0.7117 - val_loss: 1.7756


In [51]:
results["NN_model_BoW"] = model_NN_BoW.evaluate(x_test_bow, y_test.astype('category').cat.codes)[1]  # get accuracy only

1310/1310 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.6662 - loss: 1.7993


In [58]:
results['NN_model_BoW']=0.6662

In [54]:
joblib.dump(BoW, 'BoW_vectorizer.joblib')
joblib.dump(models['SVC_BoW'], 'SVC_model_BoW.joblib')
joblib.dump(models['MultinomialNB_BoW'], 'MultinomialNB_model_BoW.joblib')

['MultinomialNB_model_BoW.joblib']

In [53]:
model_NN_BoW.save("NN_model_BoW.keras")
model_NN_BoW.save("NN_model_BoW.h5")



### POS

In [57]:
models_POS={
    "SVC_POS": SVC(class_weight=class_weights_dict),
    "MultinomialNB_POS": MultinomialNB()
}

In [43]:
def pos_tag_features(text):
    tokens = word_tokenize(text)
    pos_tags = nltk.pos_tag(tokens)
    
    poscounts = {}
    for word, tag in pos_tags:
        if tag not in poscounts:
            poscounts[tag] = 1
        else:
            poscounts[tag] += 1
    return poscounts

In [88]:
x_train_POS_dic = x_train["clean_text"].apply(pos_tag_features)
x_test_POS_dic = x_test["clean_text"].apply(pos_tag_features)
# dictionary

In [ ]:
#make it vector
vec = DictVectorizer(sparse=False)  
x_train_POS = vec.fit_transform(x_train_POS_dic.tolist())
x_test_POS = vec.transform(x_test_POS_dic.tolist())


In [62]:
for model_name, model in models_POS.items():    
    model.fit(x_train_POS, y_train)
    y_pred = model.predict(x_test_POS.tolist())
    accuracy=accuracy_score(y_test, y_pred)
    print(f"Results for {model_name} with POS features: accuracy={accuracy}")
    print(classification_report(y_test, y_pred))
    print('-'*50)   
    results[model_name + "_POS"] = accuracy

Results for SVC_POS with POS features: accuracy=0.14025248788869532
                precision    recall  f1-score   support

 ENTERTAINMENT       0.16      0.29      0.21      3473
  FOOD & DRINK       0.07      0.30      0.11      1268
HEALTHY LIVING       0.10      0.25      0.14      1339
         Other       0.57      0.00      0.01     18145
     PARENTING       0.09      0.21      0.12      1758
      POLITICS       0.27      0.18      0.22      7120
  QUEER VOICES       0.08      0.05      0.06      1269
STYLE & BEAUTY       0.12      0.45      0.19      1962
        TRAVEL       0.12      0.22      0.15      1980
      WELLNESS       0.19      0.28      0.23      3589

      accuracy                           0.14     41903
     macro avg       0.18      0.23      0.14     41903
  weighted avg       0.35      0.14      0.11     41903

--------------------------------------------------
Results for MultinomialNB_POS with POS features: accuracy=0.28165047848602726
                

In [60]:
results['SVC_POS']=0.14025248788869532
results['MultinomialNB_POS']=0.28165047848602726

In [63]:
NN_model_POS = Sequential()
NN_model_POS.add(Input(shape=(x_train_POS.shape[1],)))  
NN_model_POS.add(Dense(128, activation='relu'))
NN_model_POS.add(Dense(64, activation='relu'))
NN_model_POS.add(Dense(num_classes, activation='softmax'))

NN_model_POS.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])

In [64]:
NN_model_POS.fit(x_train_POS, y_train.astype('category').cat.codes, epochs=10, batch_size=32, validation_split=0.2, callbacks=[EarlyStopping(monitor='val_loss',patience=3)])

Epoch 1/10
3045/3045 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - accuracy: 0.2760 - loss: 2.0104 - val_accuracy: 0.1615 - val_loss: 2.1987
Epoch 2/10
3045/3045 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.2940 - loss: 1.9275 - val_accuracy: 0.1510 - val_loss: 2.1794
Epoch 3/10
3045/3045 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.2976 - loss: 1.9134 - val_accuracy: 0.1715 - val_loss: 2.1552
Epoch 4/10
3045/3045 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.2974 - loss: 1.9134 - val_accuracy: 0.1262 - val_loss: 2.1863
Epoch 5/10
3045/3045 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.3018 - loss: 1.9076 - val_accuracy: 0.1403 - val_loss: 2.1636
Epoch 6/10
3045/3045 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.3016 - loss: 1.9037 - val_accuracy: 0.1576 - val_loss: 2.1802


In [65]:
results["NN_model_POS"] = NN_model_POS.evaluate(x_test_POS, y_test.astype('category').cat.codes)[1]  # get accuracy only


1310/1310 ━━━━━━━━━━━━━━━━━━━━ 1s 970us/step - accuracy: 0.3800 - loss: 1.7663


In [61]:
results['NN_model_POS']=0.3800 

In [66]:
joblib.dump(pos_tag_features, 'POS_vectorizer.joblib')
joblib.dump(models_POS['SVC_POS'], 'SVC_model_POS.joblib')
joblib.dump(models_POS['MultinomialNB_POS'], 'MultinomialNB_model_POS.joblib')

['MultinomialNB_model_POS.joblib']

### Bow + POS

In [77]:
BoW_POS_DF=  np.concatenate((x_train_bow.toarray(), x_train_POS), axis=1)
BoW_POS_DF_test=  np.concatenate((x_test_bow.toarray(), x_test_POS), axis=1)

MemoryError: Unable to allocate 18.1 GiB for an array with shape (121793, 20000) and data type int64

In [78]:
BoW_POS_df_train = hstack([x_train_bow, x_train_POS])
BoW_POS_df_test  = hstack([x_test_bow, x_test_POS])


In [49]:
models_BOW_POS={
    "SVC_BoW_POS": SVC(class_weight=class_weights_dict),    
    "MultinomialNB_BoW_POS": MultinomialNB()
}

In [50]:
for model_name, model in models_BOW_POS.items():
    model.fit(BoW_POS_df_train, y_train)
    y_pred = model.predict(BoW_POS_df_test)
    accuracy=accuracy_score(y_test, y_pred)
    print(f"Results for {model_name} with BoW + POS features: accuracy={accuracy}")
    print(classification_report(y_test, y_pred))
    print('-'*50)   
    results[model_name + "_BoW_POS"] = accuracy

Results for SVC_BoW_POS with BoW + POS features: accuracy=0.6112688828962127
                precision    recall  f1-score   support

 ENTERTAINMENT       0.47      0.74      0.58      3473
  FOOD & DRINK       0.52      0.81      0.63      1268
HEALTHY LIVING       0.21      0.49      0.30      1339
         Other       0.80      0.45      0.58     18145
     PARENTING       0.42      0.77      0.54      1758
      POLITICS       0.73      0.74      0.74      7120
  QUEER VOICES       0.72      0.67      0.69      1269
STYLE & BEAUTY       0.66      0.82      0.73      1962
        TRAVEL       0.64      0.74      0.69      1980
      WELLNESS       0.55      0.74      0.63      3589

      accuracy                           0.61     41903
     macro avg       0.57      0.70      0.61     41903
  weighted avg       0.68      0.61      0.62     41903

--------------------------------------------------
Results for MultinomialNB_BoW_POS with BoW + POS features: accuracy=0.613703076152065

In [51]:
joblib.dump(models_BOW_POS['SVC_BoW_POS'], 'SVC_model_BoW_POS.joblib')
joblib.dump(models_BOW_POS['MultinomialNB_BoW_POS'], 'MultinomialNB_model_BoW_POS.joblib')

['MultinomialNB_model_BoW_POS.joblib']

In [53]:
NN_model_BoW_POS = Sequential()
NN_model_BoW_POS.add(Input(shape=(BoW_POS_df_train.shape[1],)))
NN_model_BoW_POS.add(Dense(128, activation='relu'))
NN_model_BoW_POS.add(Dense(64, activation='relu'))  
NN_model_BoW_POS.add(Dense(num_classes, activation='softmax'))

NN_model_BoW_POS.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])

In [54]:
NN_model_BoW_POS.fit(BoW_POS_df_train, y_train.astype('category').cat.codes, epochs=10, batch_size=32, validation_split=0.2, callbacks=[EarlyStopping(monitor='val_loss',patience=3)])

Epoch 1/10
3045/3045 ━━━━━━━━━━━━━━━━━━━━ 60s 18ms/step - accuracy: 0.6099 - loss: 1.1652 - val_accuracy: 0.7701 - val_loss: 0.7331
Epoch 2/10
3045/3045 ━━━━━━━━━━━━━━━━━━━━ 58s 19ms/step - accuracy: 0.8312 - loss: 0.4868 - val_accuracy: 0.7334 - val_loss: 0.8974
Epoch 3/10
3045/3045 ━━━━━━━━━━━━━━━━━━━━ 107s 35ms/step - accuracy: 0.9213 - loss: 0.2372 - val_accuracy: 0.7508 - val_loss: 0.9505
Epoch 4/10
3045/3045 ━━━━━━━━━━━━━━━━━━━━ 73s 24ms/step - accuracy: 0.9731 - loss: 0.0898 - val_accuracy: 0.7052 - val_loss: 1.4959


In [55]:
results["NN_model_BoW_POS"] = NN_model_BoW_POS.evaluate(BoW_POS_df_test, y_test.astype('category').cat.codes)[1]  # get accuracy only

1310/1310 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.6610 - loss: 1.5333


In [62]:
results

{'SVC_BoW_POS_BoW_POS': 0.6112688828962127,
 'MultinomialNB_BoW_POS_BoW_POS': 0.6137030761520654,
 'NN_model_BoW_POS': 0.6637710928916931,
 'SVC_BoWBoW': 0.6485215855666658,
 'MultinomialNB_BoWBoW': 0.6145622031835429,
 'NN_model_BoW': 0.6662,
 'SVC_POS': 0.14025248788869532,
 'MultinomialNB_POS': 0.28165047848602726,
 'NN_model_POS': 0.38}

### NER

In [63]:
models_NER={
    "SVC_NER": SVC(class_weight=class_weights_dict),    
    "MultinomialNB_NER": MultinomialNB()
}

In [92]:
models_NER_words={
    "SVC_NER_words": SVC(class_weight=class_weights_dict),    
    "MultinomialNB_NER_words": MultinomialNB()
}

In [ ]:
def ner_features(text):
    tokens = word_tokenize(text)
    pos_tags = nltk.pos_tag(tokens)
    tree = nltk.ne_chunk(pos_tags)
    
    ner_counts = {}
    for subtree in tree:
        if hasattr(subtree, 'label'):  
            label = subtree.label()
            ner_counts[label] = ner_counts.get(label, 0) + 1
    return ner_counts

#### word only

In [102]:
def ner_words_only(text):
    doc = nlp(text)
    ner_words = [ent.label_ for ent in doc.ents]
    return ' '.join(ner_words)

In [103]:
xtrain_NER_words = x_train["clean_text"].apply(ner_words_only)
xtest_NER_words = x_test["clean_text"].apply(ner_words_only)

In [104]:
ner_word=CountVectorizer()

x_train_NER_words = ner_word.fit_transform(xtrain_NER_words)
x_test_NER_words = ner_word.transform(xtest_NER_words)

In [105]:
for model_name, model in models_NER_words.items():    
    model.fit(x_train_NER_words, y_train)
    y_pred = model.predict(x_test_NER_words)
    accuracy=accuracy_score(y_test, y_pred)
    print(f"Results for {model_name} with NER words features: accuracy={accuracy}")
    print(classification_report(y_test, y_pred))
    print('-'*50)   
    results[model_name] = accuracy

Results for SVC_NER_words with NER words features: accuracy=0.15688614180368948
                precision    recall  f1-score   support

 ENTERTAINMENT       0.21      0.36      0.26      3473
  FOOD & DRINK       0.05      0.02      0.03      1268
HEALTHY LIVING       0.06      0.59      0.11      1339
         Other       0.48      0.00      0.01     18145
     PARENTING       0.07      0.15      0.10      1758
      POLITICS       0.34      0.41      0.37      7120
  QUEER VOICES       0.07      0.02      0.03      1269
STYLE & BEAUTY       0.10      0.12      0.11      1962
        TRAVEL       0.13      0.32      0.18      1980
      WELLNESS       0.17      0.10      0.13      3589

      accuracy                           0.16     41903
     macro avg       0.17      0.21      0.13     41903
  weighted avg       0.32      0.16      0.12     41903

--------------------------------------------------
Results for MultinomialNB_NER_words with NER words features: accuracy=0.2521537837

d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


                precision    recall  f1-score   support

 ENTERTAINMENT       0.21      0.14      0.17      3473
  FOOD & DRINK       0.05      0.00      0.00      1268
HEALTHY LIVING       0.00      0.00      0.00      1339
         Other       0.47      0.26      0.34     18145
     PARENTING       0.11      0.01      0.01      1758
      POLITICS       0.18      0.68      0.29      7120
  QUEER VOICES       0.00      0.00      0.00      1269
STYLE & BEAUTY       0.22      0.00      0.00      1962
        TRAVEL       0.26      0.06      0.10      1980
      WELLNESS       0.12      0.09      0.10      3589

      accuracy                           0.25     41903
     macro avg       0.16      0.12      0.10     41903
  weighted avg       0.29      0.25      0.22     41903

--------------------------------------------------


d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [106]:
NN_model_NER_words = Sequential()
NN_model_NER_words.add(Input(shape=(x_train_NER_words.shape[1],)))  
NN_model_NER_words.add(Dense(128, activation='relu'))
NN_model_NER_words.add(Dense(64, activation='relu'))
NN_model_NER_words.add(Dense(num_classes, activation='softmax'))

NN_model_NER_words.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])

In [107]:
x_train_NER_words_dense = x_train_NER_words.toarray()
x_test_NER_words_dense  = x_test_NER_words.toarray()

NN_model_NER_words.fit(x_train_NER_words_dense, y_train.astype('category').cat.codes, epochs=10, batch_size=32, validation_split=0.2, callbacks=[EarlyStopping(monitor='val_loss',patience=3)])

Epoch 1/10
3045/3045 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - accuracy: 0.3185 - loss: 1.9395 - val_accuracy: 0.1966 - val_loss: 2.1475
Epoch 2/10
3045/3045 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.3248 - loss: 1.9009 - val_accuracy: 0.1230 - val_loss: 2.1473
Epoch 3/10
3045/3045 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - accuracy: 0.3269 - loss: 1.9011 - val_accuracy: 0.2058 - val_loss: 2.1443
Epoch 4/10
3045/3045 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.3226 - loss: 1.9020 - val_accuracy: 0.1763 - val_loss: 2.1530
Epoch 5/10
3045/3045 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.3265 - loss: 1.8950 - val_accuracy: 0.1664 - val_loss: 2.1470
Epoch 6/10
3045/3045 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.3245 - loss: 1.8973 - val_accuracy: 0.1934 - val_loss: 2.1504


In [108]:
results['NN_model_NER_words']=NN_model_NER_words.evaluate(x_test_NER_words, y_test.astype('category').cat.codes)[1]

1248/1310 ━━━━━━━━━━━━━━━━━━━━ 0s 877us/step - accuracy: 0.3967 - loss: 1.7695

InvalidArgumentError: Graph execution error:

Detected at node RaggedGather/RaggedGather defined at (most recent call last):
<stack traces unavailable>
Error in user-defined function passed to ParallelMapDatasetV2:197 transformation with iterator: Iterator::Root::Prefetch::ParallelMapV2: indices[14] = 41902 is not in [0, 41902)
	 [[{{node RaggedGather/RaggedGather}}]]
	 [[IteratorGetNext]] [Op:__inference_multi_step_on_iterator_227689]

In [66]:
x_train_NER_dic = x_train["clean_text"].apply(ner_features)
x_test_NER_dic = x_test["clean_text"].apply(ner_features)

KeyboardInterrupt: 

#### features

In [44]:
def ner_features_spacy(text):
    doc = nlp(text)
    ent_counts = {}

    for ent in doc.ents:
        label = ent.label_
        if label not in ent_counts:
            ent_counts[label] = 1
        else:
            ent_counts[label] += 1

    return ent_counts


In [71]:
x_train_NER_dic = x_train["clean_text"].apply(ner_features_spacy)
x_test_NER_dic = x_test["clean_text"].apply(ner_features_spacy)

In [74]:
x_train_NER = vec.fit_transform(x_train_NER_dic.tolist())
x_test_NER = vec.transform(x_test_NER_dic.tolist())     


In [ ]:
for model_name, model in models_NER.items():    
    model.fit(x_train_NER, y_train)
    y_pred = model.predict(x_test_NER)
    accuracy=accuracy_score(y_test, y_pred)
    print(f"Results for {model_name} with NER features: accuracy={accuracy}")
    print(classification_report(y_test, y_pred))
    print('-'*50)   
    results[model_name] = accuracy

Results for SVC_NER with NER features: accuracy=0.15688614180368948
                precision    recall  f1-score   support

 ENTERTAINMENT       0.21      0.36      0.26      3473
  FOOD & DRINK       0.05      0.02      0.03      1268
HEALTHY LIVING       0.06      0.59      0.11      1339
         Other       0.48      0.00      0.01     18145
     PARENTING       0.07      0.15      0.10      1758
      POLITICS       0.34      0.41      0.37      7120
  QUEER VOICES       0.07      0.02      0.03      1269
STYLE & BEAUTY       0.10      0.12      0.11      1962
        TRAVEL       0.13      0.32      0.18      1980
      WELLNESS       0.17      0.10      0.13      3589

      accuracy                           0.16     41903
     macro avg       0.17      0.21      0.13     41903
  weighted avg       0.32      0.16      0.12     41903

--------------------------------------------------
Results for MultinomialNB_NER with NER features: accuracy=0.25215378373863445
                

d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [76]:
results

{'SVC_BoW_POS_BoW_POS': 0.6112688828962127,
 'MultinomialNB_BoW_POS_BoW_POS': 0.6137030761520654,
 'NN_model_BoW_POS': 0.6637710928916931,
 'SVC_BoWBoW': 0.6485215855666658,
 'MultinomialNB_BoWBoW': 0.6145622031835429,
 'NN_model_BoW': 0.6662,
 'SVC_POS': 0.14025248788869532,
 'MultinomialNB_POS': 0.28165047848602726,
 'NN_model_POS': 0.38,
 'SVC_NER': 0.15688614180368948,
 'MultinomialNB_NER': 0.25215378373863445}

In [78]:
joblib.dump(models_NER['SVC_NER'], 'SVC_model_NER.joblib')
joblib.dump(models_NER['MultinomialNB_NER'], 'MultinomialNB_model_NER.joblib')


['MultinomialNB_model_NER.joblib']

In [79]:
NN_model_NER = Sequential()
NN_model_NER.add(Input(shape=(x_train_NER.shape[1],)))      
NN_model_NER.add(Dense(128, activation='relu'))
NN_model_NER.add(Dense(64, activation='relu'))
NN_model_NER.add(Dense(num_classes, activation='softmax'))

NN_model_NER.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])

In [80]:
NN_model_NER.fit(x_train_NER, y_train.astype('category').cat.codes, epochs=10, batch_size=32, validation_split=0.2, callbacks=[EarlyStopping(monitor='val_loss',patience=3)])

Epoch 1/10
3045/3045 ━━━━━━━━━━━━━━━━━━━━ 23s 2ms/step - accuracy: 0.3163 - loss: 1.9423 - val_accuracy: 0.1773 - val_loss: 2.1613
Epoch 2/10
3045/3045 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.3240 - loss: 1.9051 - val_accuracy: 0.2117 - val_loss: 2.1401
Epoch 3/10
3045/3045 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - accuracy: 0.3250 - loss: 1.9016 - val_accuracy: 0.2010 - val_loss: 2.1590
Epoch 4/10
3045/3045 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.3236 - loss: 1.9021 - val_accuracy: 0.1253 - val_loss: 2.1783
Epoch 5/10
3045/3045 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.3253 - loss: 1.8924 - val_accuracy: 0.1973 - val_loss: 2.1563


In [83]:
results["NN_model_NER"] = NN_model_NER.evaluate(x_train_NER, y_train.astype('category').cat.codes)[1] 

3807/3807 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.4546 - loss: 1.6513


In [84]:
results

{'SVC_BoW_POS_BoW_POS': 0.6112688828962127,
 'MultinomialNB_BoW_POS_BoW_POS': 0.6137030761520654,
 'NN_model_BoW_POS': 0.6637710928916931,
 'SVC_BoWBoW': 0.6485215855666658,
 'MultinomialNB_BoWBoW': 0.6145622031835429,
 'NN_model_BoW': 0.6662,
 'SVC_POS': 0.14025248788869532,
 'MultinomialNB_POS': 0.28165047848602726,
 'NN_model_POS': 0.38,
 'SVC_NER': 0.15688614180368948,
 'MultinomialNB_NER': 0.25215378373863445,
 'NN_model_NER': 0.30034565925598145}

### POS + NER

In [53]:
models_POS_NER={
    "SVC_POS_NER": SVC(class_weight=class_weights_dict),    
    "MultinomialNB_POS_NER": MultinomialNB()
}

In [47]:
x_train_POS_dic = x_train["clean_text"].apply(pos_tag_features)
x_test_POS_dic  = x_test["clean_text"].apply(pos_tag_features)


x_train_NER_dic = x_train["clean_text"].apply(ner_features_spacy)
x_test_NER_dic  = x_test["clean_text"].apply(ner_features_spacy)

vec_POS = DictVectorizer(sparse=True)
x_train_POS = vec_POS.fit_transform(x_train_POS_dic.tolist())
x_test_POS  = vec_POS.transform(x_test_POS_dic.tolist())

vec_NER = DictVectorizer(sparse=True)
x_train_NER = vec_NER.fit_transform(x_train_NER_dic.tolist())
x_test_NER  = vec_NER.transform(x_test_NER_dic.tolist())


In [51]:
x_train_POS_NER = hstack([x_train_POS, x_train_NER])
x_test_POS_NER  = hstack([x_test_POS, x_test_NER])

In [94]:
results

{'SVC_BoW_POS_BoW_POS': 0.6112688828962127,
 'MultinomialNB_BoW_POS_BoW_POS': 0.6137030761520654,
 'NN_model_BoW_POS': 0.6637710928916931,
 'SVC_BoWBoW': 0.6485215855666658,
 'MultinomialNB_BoWBoW': 0.6145622031835429,
 'NN_model_BoW': 0.6662,
 'SVC_POS': 0.14025248788869532,
 'MultinomialNB_POS': 0.28165047848602726,
 'NN_model_POS': 0.38,
 'SVC_NER': 0.15688614180368948,
 'MultinomialNB_NER': 0.25215378373863445,
 'NN_model_NER': 0.30034565925598145}

In [54]:
for model_name, model in models_POS_NER.items():
    model.fit(x_train_POS_NER, y_train)
    y_pred = model.predict(x_test_POS_NER)
    accuracy=accuracy_score(y_test, y_pred)
    print(f"Results for {model_name} with POS + NER features: accuracy={accuracy}")
    print(classification_report(y_test, y_pred))
    print('-'*50)   
    results[model_name] = accuracy

Results for SVC_POS_NER with POS + NER features: accuracy=0.20473474452903134
                precision    recall  f1-score   support

 ENTERTAINMENT       0.24      0.34      0.28      3473
  FOOD & DRINK       0.08      0.37      0.13      1268
HEALTHY LIVING       0.11      0.28      0.16      1339
         Other       0.61      0.04      0.07     18145
     PARENTING       0.10      0.25      0.15      1758
      POLITICS       0.43      0.32      0.37      7120
  QUEER VOICES       0.09      0.06      0.07      1269
STYLE & BEAUTY       0.16      0.41      0.24      1962
        TRAVEL       0.16      0.31      0.21      1980
      WELLNESS       0.23      0.46      0.31      3589

      accuracy                           0.20     41903
     macro avg       0.22      0.28      0.20     41903
  weighted avg       0.40      0.20      0.18     41903

--------------------------------------------------
Results for MultinomialNB_POS_NER with POS + NER features: accuracy=0.34725437319523

In [57]:
NN_model_POS_NER = Sequential()
NN_model_POS_NER.add(Input(shape=(x_train_POS_NER.shape[1],)))
NN_model_POS_NER.add(Dense(128, activation='relu')) 
NN_model_POS_NER.add(Dense(64, activation='relu'))
NN_model_POS_NER.add(Dense(num_classes, activation='softmax'))

NN_model_POS_NER.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])

In [58]:
NN_model_POS_NER.fit(x_train_POS_NER, y_train.astype('category').cat.codes, epochs=10, batch_size=32, validation_split=0.2, callbacks=[EarlyStopping(monitor='val_loss',patience=3)])

Epoch 1/10
3045/3045 ━━━━━━━━━━━━━━━━━━━━ 29s 2ms/step - accuracy: 0.3186 - loss: 1.9302 - val_accuracy: 0.2930 - val_loss: 1.9754
Epoch 2/10
3045/3045 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.3499 - loss: 1.8039 - val_accuracy: 0.2666 - val_loss: 1.9985
Epoch 3/10
3045/3045 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.3521 - loss: 1.7895 - val_accuracy: 0.2343 - val_loss: 2.0221
Epoch 4/10
3045/3045 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.3584 - loss: 1.7753 - val_accuracy: 0.2995 - val_loss: 1.9671
Epoch 5/10
3045/3045 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.3627 - loss: 1.7725 - val_accuracy: 0.2604 - val_loss: 2.0150
Epoch 6/10
3045/3045 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.3628 - loss: 1.7643 - val_accuracy: 0.2603 - val_loss: 1.9979
Epoch 7/10
3045/3045 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.3646 - loss: 1.7569 - val_accuracy: 0.2546 - val_loss: 2.0220


In [59]:
results["NN_model_POS_NER"] = NN_model_POS_NER.evaluate(x_test_POS_NER, y_test.astype('category').cat.codes)[1]

1310/1310 ━━━━━━━━━━━━━━━━━━━━ 1s 881us/step - accuracy: 0.4057 - loss: 1.6627


# HMM

In [86]:
hmm_BOW = GaussianHMM(n_components=num_classes, covariance_type="diag", n_iter=100, random_state=42)
hmm_POS = GaussianHMM(n_components=num_classes, covariance_type="diag", n_iter=100, random_state=42)
hmm_BOW_POS = GaussianHMM(n_components=num_classes, covariance_type="diag", n_iter=100, random_state=42)
hmm_NER = GaussianHMM(n_components=num_classes, covariance_type="diag", n_iter=100, random_state=42)
hmm_POS_NER = GaussianHMM(n_components=num_classes, covariance_type="diag", n_iter=100, random_state=42)

In [79]:
from sklearn.decomposition import PCA

pca = PCA(n_components=50)
x_train_bow_pca = pca.fit_transform(x_train_bow.toarray())
x_test_bow_pca = pca.transform(x_test_bow.toarray())

MemoryError: Unable to allocate 18.1 GiB for an array with shape (121793, 20000) and data type int64

In [118]:
from sklearn.decomposition import TruncatedSVD

svd_bow = TruncatedSVD(n_components=50)
x_train_bow_svd = svd_bow.fit_transform(x_train_bow)
x_test_bow_svd = svd_bow.transform(x_test_bow)

In [119]:
svd_pos = TruncatedSVD(n_components=20)
x_train_POS_svd = svd_pos.fit_transform(x_train_POS)
x_test_POS_svd = svd_pos.transform(x_test_POS)

In [120]:
svd_bow_pos = TruncatedSVD(n_components=70)
BoW_POS_svd_train = svd_bow_pos.fit_transform(BoW_POS_df_train)
BoW_POS_svd_test = svd_bow_pos.transform(BoW_POS_df_test)

In [121]:
svd_pos_ner = TruncatedSVD(n_components=30)
x_train_POS_NER_svd = svd_pos_ner.fit_transform(x_train_POS_NER)
x_test_POS_NER_svd = svd_pos_ner.transform(x_test_POS_NER)  

In [122]:
hmm_BOW.fit(x_test_bow_svd)
hmm_POS.fit(x_train_POS_svd)

Even though the 'startprob_' attribute is set, it will be overwritten during initialization because 'init_params' contains 's'
Even though the 'transmat_' attribute is set, it will be overwritten during initialization because 'init_params' contains 't'
Even though the 'means_' attribute is set, it will be overwritten during initialization because 'init_params' contains 'm'
Even though the 'covars_' attribute is set, it will be overwritten during initialization because 'init_params' contains 'c'
Even though the 'startprob_' attribute is set, it will be overwritten during initialization because 'init_params' contains 's'
Even though the 'transmat_' attribute is set, it will be overwritten during initialization because 'init_params' contains 't'
Even though the 'means_' attribute is set, it will be overwritten during initialization because 'init_params' contains 'm'
Even though the 'covars_' attribute is set, it will be overwritten during initialization because 'init_params' contains 'c'


GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [109]:
hmm_BOW_POS.fit(BoW_POS_svd_train)

Even though the 'startprob_' attribute is set, it will be overwritten during initialization because 'init_params' contains 's'
Even though the 'transmat_' attribute is set, it will be overwritten during initialization because 'init_params' contains 't'
Even though the 'means_' attribute is set, it will be overwritten during initialization because 'init_params' contains 'm'
Even though the 'covars_' attribute is set, it will be overwritten during initialization because 'init_params' contains 'c'


GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [110]:
hmm_NER.fit(x_train_NER.toarray())  # nfeatures = 18 < n components in svd 


Model is not converging.  Current: 10563647.441249046 is not greater than 10563647.445636675. Delta is -0.00438762828707695


GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [111]:

hmm_POS_NER.fit(x_train_POS_NER_svd)

GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [114]:
y_train_enc = y_train.astype('category').cat.codes
y_test_enc  = y_test.astype('category').cat.codes


In [136]:
y_pred_hmm_BOW = hmm_BOW.predict(x_test_bow_svd)
accuracy_hmm_BOW = accuracy_score(y_test_enc, y_pred_hmm_BOW)   
traing_accuracy_hmm_BOW = accuracy_score(y_train_enc, hmm_BOW.predict(x_train_bow_svd))
print(f"Training HMM BoW Accuracy: {traing_accuracy_hmm_BOW}")
print(f"HMM BoW Accuracy: {accuracy_hmm_BOW}")
results['HMM_BoW']=accuracy_hmm_BOW

Training HMM BoW Accuracy: 0.07402724294499684
HMM BoW Accuracy: 0.0729542037562943


In [137]:
y_pred_hmm_POS = hmm_POS.predict(x_test_POS_svd)
accuracy_hmm_POS = accuracy_score(y_test_enc, y_pred_hmm_POS)
traing_accuracy_hmm_POS = accuracy_score(y_train_enc, hmm_POS.predict(x_train_POS_svd))
print(f"Training HMM POS Accuracy: {traing_accuracy_hmm_POS}")
print(f"HMM POS Accuracy: {accuracy_hmm_POS}")
results['HMM_POS']=accuracy_hmm_POS

Training HMM POS Accuracy: 0.09122855993365793
HMM POS Accuracy: 0.08421831372455432


In [138]:
y_pred_hmm_bow_pos = hmm_BOW_POS.predict(BoW_POS_svd_test)
accuracy_hmm_bow_pos = accuracy_score(y_test_enc, y_pred_hmm_bow_pos)
traing_accuracy_hmm_bow_pos = accuracy_score(y_train_enc, hmm_BOW_POS.predict(BoW_POS_svd_train))
print(f"Training HMM BoW + POS Accuracy: {traing_accuracy_hmm_bow_pos}")
print(f"HMM BoW + POS Accuracy: {accuracy_hmm_bow_pos}")
results['HMM_BoW_POS']=accuracy_hmm_bow_pos

Training HMM BoW + POS Accuracy: 0.09521893704892728
HMM BoW + POS Accuracy: 0.0889196477579171


In [139]:
y_pred_hmm_ner = hmm_NER.predict(x_test_NER.toarray())
accuracy_hmm_ner = accuracy_score(y_test_enc, y_pred_hmm_ner)
traing_accuracy_hmm_ner = accuracy_score(y_train_enc, hmm_NER.predict(x_train_NER.toarray()))
print(f"Training HMM NER Accuracy: {traing_accuracy_hmm_ner}")
print(f"HMM NER Accuracy: {accuracy_hmm_ner}")
results['HMM_NER']=accuracy_hmm_ner

Training HMM NER Accuracy: 0.05436273020616948
HMM NER Accuracy: 0.03928119705033053


In [140]:
y_pred_hmm_pos_ner = hmm_POS_NER.predict(x_test_POS_NER_svd)
accuracy_hmm_pos_ner = accuracy_score(y_test_enc, y_pred_hmm_pos_ner)
traing_accuracy_hmm_pos_ner = accuracy_score(y_train_enc, hmm_POS_NER.predict(x_train_POS_NER_svd))
print(f"Training HMM POS + NER Accuracy: {traing_accuracy_hmm_pos_ner}")
print(f"HMM POS + NER Accuracy: {accuracy_hmm_pos_ner}")
results['HMM_POS_NER']=accuracy_hmm_pos_ner

Training HMM POS + NER Accuracy: 0.07716371220020855
HMM POS + NER Accuracy: 0.06512660191394411


In [141]:
results

{'SVC_BoW_POS_BoW_POS': 0.6112688828962127,
 'MultinomialNB_BoW_POS_BoW_POS': 0.6137030761520654,
 'NN_model_BoW_POS': 0.6637710928916931,
 'SVC_BoWBoW': 0.6485215855666658,
 'MultinomialNB_BoWBoW': 0.6145622031835429,
 'NN_model_BoW': 0.6662,
 'SVC_POS': 0.14025248788869532,
 'MultinomialNB_POS': 0.28165047848602726,
 'NN_model_POS': 0.38,
 'SVC_NER': 0.15688614180368948,
 'MultinomialNB_NER': 0.25215378373863445,
 'NN_model_NER': 0.30034565925598145,
 'SVC_POS_NER': 0.20473474452903134,
 'MultinomialNB_POS_NER': 0.3472543731952366,
 'NN_model_POS_NER': 0.4070591628551483,
 'SVC_NER_words': 0.15688614180368948,
 'MultinomialNB_NER_words': 0.25215378373863445,
 'HMM_BoW': 0.0729542037562943,
 'HMM_POS': 0.08421831372455432,
 'HMM_BoW_POS': 0.0889196477579171,
 'HHM_BOW': 0.08312054029544425,
 'HMM_BOW_POS': 0.08887191847839057,
 'HMM_NER': 0.03928119705033053,
 'HMM_POS_NER': 0.06512660191394411}

# Save the results

In [143]:
with open('results_dict.pkl', 'wb') as f:
    pickle.dump(results, f)


In [52]:
with open('results_dict.pkl', 'rb') as f:
    results = pickle.load(f)
print(results)

{'SVC_BoW_POS_BoW_POS': 0.6112688828962127, 'MultinomialNB_BoW_POS_BoW_POS': 0.6137030761520654, 'NN_model_BoW_POS': 0.6637710928916931, 'SVC_BoWBoW': 0.6485215855666658, 'MultinomialNB_BoWBoW': 0.6145622031835429, 'NN_model_BoW': 0.6662, 'SVC_POS': 0.14025248788869532, 'MultinomialNB_POS': 0.28165047848602726, 'NN_model_POS': 0.38, 'SVC_NER': 0.15688614180368948, 'MultinomialNB_NER': 0.25215378373863445, 'NN_model_NER': 0.30034565925598145}


In [ ]:
final1={'SVC_BoW_POS_BoW_POS': 0.6112688828962127,
 'MultinomialNB_BoW_POS_BoW_POS': 0.6137030761520654,
 'NN_model_BoW_POS': 0.6637710928916931,
 'SVC_BoWBoW': 0.6485215855666658,
 'MultinomialNB_BoWBoW': 0.6145622031835429,
 'NN_model_BoW': 0.6662,
 'SVC_POS': 0.14025248788869532,
 'MultinomialNB_POS': 0.28165047848602726,
 'NN_model_POS': 0.38,
 'SVC_NER': 0.15688614180368948,
 'MultinomialNB_NER': 0.25215378373863445,
 'NN_model_NER': 0.30034565925598145,
 'SVC_POS_NER': 0.20473474452903134,
 'MultinomialNB_POS_NER': 0.3472543731952366,
 'NN_model_POS_NER': 0.4070591628551483,
 'SVC_NER_words': 0.15688614180368948,
 'MultinomialNB_NER_words': 0.25215378373863445,
 'HMM_BoW': 0.0729542037562943,
 'HMM_POS': 0.08421831372455432,
 'HMM_BoW_POS': 0.0889196477579171,
 'HHM_BOW': 0.08312054029544425,
 'HMM_BOW_POS': 0.08887191847839057,
 'HMM_NER': 0.03928119705033053,
 'HMM_POS_NER': 0.06512660191394411}

In [ ]:
with open("final1.txt", "w", encoding="utf-8") as f:
    for key, value in final1.items():
        f.write(f"{key}: {value}\n")

In [ ]:
joblib.dump(final1,'final1')